# ResumeModel v4 — Training & Inference Notebook
This notebook implements a runnable V4 training pipeline, simple resume-job matching, skill extraction stubs, and export steps.

Sections:
1. Setup
2. Data loading
3. Preprocessing
4. Feature extraction
5. Model training
6. Evaluation
7. Save artifacts
8. Matching & skill extraction stubs
9. Inference demo
10. Next steps

In [ ]:
# Setup / Requirements
from sklearn.metrics import accuracy_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib
import pandas as pd
import numpy as np
import sys
import platform
import time
import os
import re
from pathlib import Path

print('Python', sys.version.split()[0])
print('Platform:', platform.platform())

# Core data + ML libs


# spaCy import — require en_core_web_sm
try:
    import spacy
    _ = spacy.load('en_core_web_sm')
except Exception as e:
    print('spaCy or model not available:', e)
    print('If needed, install with: python -m pip install spacy && python -m spacy download en_core_web_sm')
    # Import anyway so helper functions can still be defined
    import spacy

print('All imports done')

In [ ]:
# Data loading
from pathlib import Path
base = Path('Dataset')

print('Reading datasets from', base)

df1 = pd.read_csv(base / 'UpdatedResumeDataSet.csv')
print('df1', df1.shape)

df2 = pd.read_csv(base / 'resume_dataset.csv')
print('df2', df2.shape)

df3_raw = pd.read_csv(base / 'Resume.csv')
# Map categories in df3_raw if needed
category_map = {
    'ACCOUNTANT': 'Accountant',
    'ADVOCATE': 'Advocate',
    'AGRICULTURE': 'Agriculture',
    'APPAREL': 'Apparel',
    'ARTS': 'Arts',
    'AUTOMOBILE': 'Automobile',
    'AVIATION': 'Aviation',
    'BANKING': 'Banking',
    'BPO': 'BPO',
    'BUSINESS-DEVELOPMENT': 'Business Development',
    'CHEF': 'Chef',
    'CONSTRUCTION': 'Construction',
    'CONSULTANT': 'Consultant',
    'DESIGNER': 'Designer',
    'DIGITAL-MEDIA': 'Digital Media',
    'ENGINEERING': 'Engineering',
    'FINANCE': 'Finance',
    'FITNESS': 'Health and fitness',
    'HEALTHCARE': 'Healthcare',
    'HR': 'HR',
    'INFORMATION-TECHNOLOGY': 'Information Technology',
    'PUBLIC-RELATIONS': 'Public Relations',
    'SALES': 'Sales',
    'TEACHER': 'Teacher',
}

df3 = pd.DataFrame({
    'Category': df3_raw['Category'].map(category_map),
    'Resume': df3_raw['Resume_str']
})

df3 = df3.dropna(subset=['Category', 'Resume'])

df = pd.concat([df1, df2, df3], ignore_index=True)
print('Combined shape before cleaning:', df.shape)

# Basic cleaning
df = df.drop_duplicates(subset=['Resume'])
df = df.dropna(subset=['Category', 'Resume'])
df = df[df['Resume'].str.strip() != '']

# Filter small categories
cat_counts = df['Category'].value_counts()
valid_cats = cat_counts[cat_counts >= 5].index.tolist()
df = df[df['Category'].isin(valid_cats)].reset_index(drop=True)

print('Final combined dataset:', df.shape)
print('Categories:', df['Category'].nunique())

In [ ]:
# Preprocessing helpers
import re
import spacy
nlp = spacy.load('en_core_web_sm')


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'\bRT\b|\bcc\b', ' ', text)
    text = re.sub(r'#\S+', ' ', text)
    text = re.sub(r'@\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def spacy_preprocess(text: str) -> str:
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space and len(token.text) > 1
    ]
    return ' '.join(tokens)


print('Preprocessing functions ready')

In [ ]:
# Apply preprocessing (warn: can be slow for large datasets)
print('Cleaning raw text...')
df['Cleaned_Resume'] = df['Resume'].apply(clean_text)
print('Running spaCy preprocessing...')
start = time.time()
df['Processed_Resume'] = df['Cleaned_Resume'].apply(spacy_preprocess)
print('Preprocessing done in', int(time.time()-start), 's')

# Quick sanity
print(df[['Processed_Resume', 'Category']].head(3).to_dict())

In [ ]:
# Feature extraction: TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000, sublinear_tf=True, ngram_range=(1, 3))
X = tfidf.fit_transform(df['Processed_Resume'])
print('TF-IDF shape:', X.shape)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['Category'])
print('Classes:', list(le.classes_)[:10], '... total', len(le.classes_))

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train/test sizes:', X_train.shape[0], X_test.shape[0])

# Model building
base_clf = SGDClassifier(loss='hinge', class_weight='balanced',
                         max_iter=1000, tol=1e-3, random_state=42)
calib = CalibratedClassifierCV(base_clf, cv=3, method='sigmoid')
model = OneVsRestClassifier(calib)

print('Training model...')
start = time.time()
model.fit(X_train, y_train)
print('Training finished in', int(time.time()-start), 's')

In [ ]:
# Evaluation
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print('Accuracy:', acc)
print('\nClassification report:\n')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Save artifacts
import json
out_dir = Path('..') / 'v4'
out_dir = out_dir.resolve()
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, out_dir / 'model.pkl')
joblib.dump(tfidf, out_dir / 'tfidf.pkl')
joblib.dump(le, out_dir / 'encoder.pkl')

manifest = {
    'model_path': str(out_dir / 'model.pkl'),
    'tfidf_path': str(out_dir / 'tfidf.pkl'),
    'encoder_path': str(out_dir / 'encoder.pkl'),
    'trained_on': time.ctime(),
    'num_samples': int(len(df)),
    'num_classes': int(len(le.classes_))
}

with open(out_dir / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('Artifacts saved to', out_dir)
print(manifest)

In [ ]:
# Matching stub (TF-IDF cosine similarity)
from sklearn.metrics.pairwise import cosine_similarity

# Build a TF-IDF for job descriptions (simple reuse of same vectorizer for demo)
job_tfidf = tfidf


def match_score(resume_text: str, job_text: str) -> dict:
    r = spacy_preprocess(clean_text(resume_text))
    j = spacy_preprocess(clean_text(job_text))
    Xr = tfidf.transform([r])
    Xj = tfidf.transform([j])
    score = float(cosine_similarity(Xr, Xj)[0, 0])
    # top overlapping tokens (naive)
    rv = set(r.split())
    jv = set(j.split())
    overlap = list(rv & jv)[:10]
    return {'score': score, 'overlap': overlap}


print('Matching stub ready')

In [ ]:
# Skill extraction stub
# Very small demo skill list
SKILL_DICT = ['python', 'java', 'sql', 'javascript',
              'react', 'docker', 'aws', 'c++', 'c#', 'linux']


def extract_skills(text: str) -> list:
    t = spacy_preprocess(clean_text(text))
    toks = set(t.split())
    found = [s for s in SKILL_DICT if s in toks]
    # augment with noun chunks
    doc = nlp(text)
    for nc in doc.noun_chunks:
        tok = nc.text.strip().lower()
        if tok in SKILL_DICT and tok not in found:
            found.append(tok)
    return found


print('Skill extraction stub ready')

In [ ]:
# Inference demo
sample_resume = df['Resume'].iloc[0] if len(
    df) > 0 else 'Experienced software engineer with Python, AWS and Docker.'
sample_job = 'Looking for a backend engineer with experience in Python, Docker, AWS and SQL.'

print('Predict category:')
proc = spacy_preprocess(clean_text(sample_resume))
Xr = tfidf.transform([proc])
pred = model.predict(Xr)[0]
cat = le.inverse_transform([pred])[0]
print('Category:', cat)

print('\nMatch demo:')
print(match_score(sample_resume, sample_job))

print('\nExtracted skills:')
print(extract_skills(sample_resume))

## Next steps / Export

1. Move `FullStackApp/v4/model.pkl`, `tfidf.pkl`, `encoder.pkl` into your API service and implement `preprocess.clean_text` + `preprocess.spacy_preprocess` identically.
2. Scaffold FastAPI endpoints in `FullStackApp/backend/app/api.py` using the blueprint in `Report/V4_implementation_blueprint.md`.
3. Replace TF-IDF matching with a dual-encoder (sentence-transformers) for production ranking.
4. Add a robust skill dictionary / ontology and train a NER-based skill extractor for higher recall/precision.

Export to script:

```bash
python -m nbconvert --to script FullStackApp/train/ResumeModel_v4.ipynb
```

CI snippet to run quick smoke test (GitHub Actions job):

```yaml
- name: Run V4 smoke notebook
  run: |
    python -m pip install -r FullStackApp/backend/requirements.txt
    python -m nbconvert --to script FullStackApp/train/ResumeModel_v4.ipynb
    python FullStackApp/train/ResumeModel_v4.py
```

That's it — the notebook implements the core V4 pipeline and quick stubs. Run the cells top-to-bottom to train and export artifacts.